Candidate generation

In [1]:
import json
from pathlib import Path

import polars as pl

# candidate generation: pipelines/generate_candidates.p
output_dir = Path("data/processed/candidates/validation")
manifest = json.loads((output_dir / "_manifest.json").read_text())
customers = pl.read_parquet("data/processed/customers.parquet")

manifest["window_start"], manifest["chunk_size"], manifest["n_chunks"], manifest["als_random_state"], manifest["wall_time_s"], manifest["peak_rss_gb"]

('2020-09-16', 40000, 35, 42, 269.6, 4.5)

In [2]:
strategy_totals = pl.DataFrame(
    {
        "strategy": [
            "repurchase",
            "colour_variant",
            "recent_popularity",
            "segment_popularity",
            "als",
        ],
        "candidate_rows": [
            manifest["totals"]["repurchase"],
            manifest["totals"]["colour_variant"],
            manifest["totals"]["recent_popularity"],
            manifest["totals"]["segment_popularity"],
            manifest["totals"]["als"],
        ],
    }
)
strategy_totals

strategy,candidate_rows
str,i64
"""repurchase""",10420276
"""colour_variant""",22079574
"""recent_popularity""",27439600
"""segment_popularity""",27439600
"""als""",40701270


In [3]:
combined = pl.scan_parquet(str(output_dir / "part-*.parquet"))
row_count = combined.select(pl.len()).collect(engine="streaming").item()
customer_count = combined.select(pl.col("customer_id").n_unique()).collect(engine="streaming").item()

row_count, customer_count, customers.height

(105365317, 1371980, 1371980)

In [4]:
per_customer = (
    combined.group_by("customer_id")
    .agg(pl.len().alias("n_candidates"))
    .select("n_candidates")
    .collect(engine="streaming")
)
per_customer["n_candidates"].describe()

statistic,value
str,f64
"""count""",1.37198e6
"""null_count""",0.0
"""mean""",76.797998
"""std""",18.208676
"""min""",25.0
"""25%""",59.0
"""50%""",77.0
"""75%""",95.0
"""max""",100.0


Recall@100

In [5]:
recall_manifest = json.loads(Path("data/processed/candidates/_recall_manifest.json").read_text())

pl.DataFrame([{k: recall_manifest[k] for k in ["window_start", "window_end", "true_pairs", "pool_hits", "pool_recall_at_100"]}])

window_start,window_end,true_pairs,pool_hits,pool_recall_at_100
str,str,i64,i64,f64
"""2020-09-16""","""2020-09-22""",213728,21614,0.101129


Recall@100 per strategy

In [6]:
pl.DataFrame(recall_manifest["per_strategy"])

strategy,recall_at_100,marginal_contribution
str,f64,f64
"""repurchase""",0.032995,0.026089
"""colour_variant""",0.021116,0.013513
"""recent_popularity""",0.033884,0.007453
"""segment_popularity""",0.035929,0.00904
"""als""",0.014224,0.011187


Recall@100 by recency bucket

In [7]:
pl.DataFrame(recall_manifest["per_recency_bucket"])

recency_bucket,true_pairs,recall_at_100
str,i64,f64
"""last 1 week""",37546,0.185239
"""last 4 weeks""",56771,0.105899
"""last 12 weeks""",57893,0.080251
"""last 52 weeks""",38767,0.076715
"""more than 52 weeks ago""",6536,0.057834
"""never""",16215,0.040025
